# RecallRadar Canada | Databricks analysis

**Author:** Hazim Ali  
**Source:** [Government of Canada Recalls and Safety Alerts](https://open.canada.ca/data/en/dataset/d38de914-c94c-429b-8ab1-8776c31643e3)  
**Licence:** [Open Government Licence – Canada](https://open.canada.ca/en/open-government-licence-canada)

This notebook ingests the official English JSON feed, validates notice IDs and URLs, cleans dates, maps broad product sectors, and builds analytical views. The source field is **Last updated**, not first publication or incident date. Counts are of source notices, not affected products or injuries. The public app and its exact snapshot-building code are in the [GitHub repository](https://github.com/HazimAli07/recallradar-canada).

Run each cell in order on Databricks compute with internet access. Leave `as_of_utc` blank for the current UTC date, or enter the public snapshot date to reproduce its time window. This notebook uses temporary views and makes no persistent table by default.


In [ ]:
import html
import json
import re
from datetime import date, datetime, timezone
from time import time_ns
from urllib.request import urlopen
from pyspark.sql import functions as F, types as T

SOURCE = "https://recalls-rappels.canada.ca/sites/default/files/opendata-donneesouvertes/HCRSAMOpenData.json"
dbutils.widgets.text("as_of_utc", "")
as_of_override = dbutils.widgets.get("as_of_utc").strip()
AS_OF = date.fromisoformat(as_of_override) if as_of_override else datetime.now(timezone.utc).date()

# A unique query avoids a briefly stale copy served by the publisher CDN.
with urlopen(f"{SOURCE}?cb={time_ns()}", timeout=60) as response:
    source_rows = json.load(response)

if not isinstance(source_rows, list):
    raise ValueError("Expected a list of recall notices")

print(f"Source records: {len(source_rows):,}; as-of UTC date: {AS_OF}")


In [ ]:
def clean(value):
    text = html.unescape(str(value or ""))
    text = re.sub(r"<[^>]*>", " ", text)
    return " ".join(text.replace("\u200b", "").replace("\ufeff", "").split())

fields = ["NID", "Title", "URL", "Organization", "Product", "Issue", "Category", "Recall class", "Last updated", "Archived"]
rows = [tuple(clean(row.get(field)) for field in fields) for row in source_rows]
schema = T.StructType([T.StructField(field, T.StringType(), True) for field in fields])
bronze = spark.createDataFrame(rows, schema=schema)
bronze.createOrReplaceTempView("recallradar_bronze")
display(bronze.select("NID", "Title", "Organization", "Last updated").limit(10))


In [ ]:
quality = bronze.agg(
    F.count("*").alias("source_rows"),
    F.countDistinct("NID").alias("distinct_ids"),
    F.sum(F.when(F.expr("try_cast(`Last updated` as date)").isNull(), 1).otherwise(0)).alias("missing_or_invalid_dates"),
    F.sum(F.when(~F.col("URL").startswith("https://recalls-rappels.canada.ca/"), 1).otherwise(0)).alias("nonofficial_urls"),
)
display(quality)


In [ ]:
sector_map = (
    F.when(F.col("Organization") == "TC", "Vehicles")
     .when(F.col("Organization") == "CFIA", "Food")
     .when(F.col("Organization") == "Medical devices", "Medical devices")
     .when(F.col("Organization") == "Consumer product safety", "Consumer products")
     .when(F.col("Organization").isin("Drugs and health products", "Marketed health products"), "Health products")
     .when(F.col("Organization") == "Controlled substances and cannabis", "Cannabis")
     .when((F.col("Organization") == "Communications and Public Affairs Branch") & F.col("Category").isin("Food"), "Food")
     .when((F.col("Organization") == "Communications and Public Affairs Branch") & F.col("Category").isin("Medical devices", "Radiology"), "Medical devices")
     .when((F.col("Organization") == "Communications and Public Affairs Branch") & F.col("Category").isin("Drugs", "Health products", "Natural health products", "Biologic or vaccine", "Radiopharmaceuticals", "Drugs - Natural health products"), "Health products")
     .when((F.col("Organization") == "Communications and Public Affairs Branch") & F.col("Category").isin("Beauty and personal care - Specialized products", "Arts, crafts and needlework - Toys and games", "Appliances - Outdoor living - Specialized products"), "Consumer products")
     .otherwise("Other")
)

silver = (
    bronze.filter(F.col("NID").rlike(r"^[0-9]+$"))
          .filter(F.col("URL").startswith("https://recalls-rappels.canada.ca/"))
          .dropDuplicates(["NID"])
          .withColumn("updated_date", F.expr("try_cast(`Last updated` as date)"))
          .withColumn("sector", sector_map)
          .withColumn("archived_in_source", F.col("Archived") == "1")
)
silver.createOrReplaceTempView("recallradar_silver")

# Approximate 36-month scope, matching the public explorer.
as_of_date = F.lit(AS_OF.isoformat()).cast("date")
gold = silver.filter(F.col("updated_date").between(F.date_sub(as_of_date, 36 * 31), as_of_date))
gold.createOrReplaceTempView("recallradar_gold")
print(f"Dated notices in window: {gold.count():,}")


In [ ]:
# Which broad sectors have the most recently updated notices?
display(spark.sql(f"""
SELECT sector,
       COUNT(*) AS notice_count,
       SUM(CASE WHEN updated_date >= date_sub(DATE '{AS_OF.isoformat()}', 6) THEN 1 ELSE 0 END) AS updated_last_7_days,
       COUNT(DISTINCT Category) AS source_categories
FROM recallradar_gold
GROUP BY sector
ORDER BY notice_count DESC
"""))


In [ ]:
# Monthly pattern: count last updates, not new incidents or harm.
display(spark.sql(f"""
SELECT date_format(updated_date, 'yyyy-MM') AS update_month,
       sector,
       COUNT(*) AS notice_count
FROM recallradar_gold
WHERE updated_date >= add_months(trunc(DATE '{AS_OF.isoformat()}', 'MM'), -12)
  AND updated_date < trunc(DATE '{AS_OF.isoformat()}', 'MM')
GROUP BY date_format(updated_date, 'yyyy-MM'), sector
ORDER BY update_month, sector
"""))


In [ ]:
# Most common source issue labels by sector.
display(spark.sql("""
SELECT sector, Issue AS source_issue_label, COUNT(*) AS notice_count
FROM recallradar_gold
WHERE Issue IS NOT NULL AND trim(Issue) <> ''
GROUP BY sector, Issue
ORDER BY notice_count DESC
LIMIT 30
"""))


## Interpretation and next steps

- The date in these charts is the publisher's **last update**, so an older notice can enter a recent period after a revision.
- Source recall classes differ across sectors; this notebook does not combine them into a single severity score.
- `Archived` is the publisher's flag. It does not establish that a product is safe or that a problem is resolved.
- Records without valid update dates remain in the Bronze/Silver views but are excluded from the time-based Gold view.
- Any operational decision should open and read the original notice. This notebook provides exploration, not safety instructions.

**Optional extension:** schedule the notebook in your own Databricks workspace and write Gold to a Delta table after confirming your catalog permissions. The GitHub project includes a dependency-free public app and an independent scheduled refresh.
